#simle piple for the heathcare system

In [0]:
# healthcare_monitoring.py

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

# Create Spark session
spark = SparkSession.builder \
    .appName("HealthcareMonitoring") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

# Define schema for incoming JSON data
schema = StructType([
    StructField("patient_id", StringType(), True),
    StructField("heart_rate", IntegerType(), True),
    StructField("spo2", DoubleType(), True),
    StructField("blood_pressure", StringType(), True)
])

# Read stream from Kafka
raw_df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("subscribe", "patient_vitals") \
    .load()

# Convert Kafka value to string
json_df = raw_df.selectExpr("CAST(value AS STRING) as json_value")

# Parse JSON
vitals_df = json_df.select(
    from_json(col("json_value"), schema).alias("data")
).select("data.*")

# Data cleaning
clean_df = vitals_df.dropna()

# Detect abnormal conditions
alerts_df = clean_df.filter(
    (col("heart_rate") > 120) |
    (col("heart_rate") < 50) |
    (col("spo2") < 90)
)

# Output alerts
query = alerts_df.writeStream \
    .outputMode("append") \
    .format("console") \
    .option("truncate", "false") \
    .start()

query.awaitTermination()